In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anjalibhegam/multimodal-sports-injury-dataset/README.md
/kaggle/input/datasets/anjalibhegam/multimodal-sports-injury-dataset/multimodal_sports_injury_dataset.csv


In [2]:
import glob, time, os
import numpy as np
import pandas as pd

csv_path = glob.glob("/kaggle/input/**/*.csv", recursive=True)[0]
raw = pd.read_csv(csv_path)

print("Loaded:", csv_path)
print("Rows, Columns:", raw.shape)

Loaded: /kaggle/input/datasets/anjalibhegam/multimodal-sports-injury-dataset/multimodal_sports_injury_dataset.csv
Rows, Columns: (15420, 29)


In [3]:
keep_cols = ["athlete_id", "session_id",
             "training_duration", "training_intensity", "training_load",
             "heart_rate", "sleep_quality", "recovery_score", "stress_level",
             "fatigue_index"]

df = raw[keep_cols].copy()
print(df.shape)
print(df.dtypes)
df.head()

(15420, 10)
athlete_id              int64
session_id              int64
training_duration     float64
training_intensity    float64
training_load         float64
heart_rate            float64
sleep_quality         float64
recovery_score        float64
stress_level          float64
fatigue_index         float64
dtype: object


,athlete_id,session_id,training_duration,training_intensity,training_load,heart_rate,sleep_quality,recovery_score,stress_level,fatigue_index
0,1,1,79.106564,5.678475,548.417962,NaN,5.466370,77.899691,0.375825,45.028687
1,1,2,116.954654,NaN,538.043815,40.000000,6.899090,74.845323,0.279165,42.254717
2,1,3,96.354742,6.643242,647.784339,76.174609,9.574152,83.538316,0.334061,61.870056
3,1,4,97.619130,6.194427,336.553431,100.543724,7.614297,79.966618,0.218766,50.268845
4,1,5,83.155527,5.835804,496.076352,78.001235,5.631979,56.945979,0.484086,29.233575


In [4]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print("Fully duplicated rows:", df.duplicated().sum())

print("\n=== DUPLICATE (athlete, session) PAIRS ===")
print("Repeated athlete+session pairs:", df.duplicated(subset=["athlete_id", "session_id"]).sum())

print("\n=== VALUE RANGES (outliers are NOT deleted) ===")
print(df.describe().T[["min", "mean", "max"]].round(2))

=== MISSING VALUES ===
athlete_id              0
session_id              0
training_duration       0
training_intensity    632
training_load           0
heart_rate            354
sleep_quality         493
recovery_score          0
stress_level            0
fatigue_index           0
dtype: int64

=== DUPLICATE ROWS ===
Fully duplicated rows: 0

=== DUPLICATE (athlete, session) PAIRS ===
Repeated athlete+session pairs: 0

=== VALUE RANGES (outliers are NOT deleted) ===
                       min    mean      max
athlete_id            1.00   78.55   156.00
session_id            1.00   49.93   101.00
training_duration    30.00   87.79   180.00
training_intensity    2.00    6.40    10.00
training_load       150.00  670.53  2632.64
heart_rate           40.00   72.69   159.27
sleep_quality         1.16    5.82    10.00
recovery_score        8.90   55.20    98.00
stress_level          0.10    0.42     0.95
fatigue_index        15.00   59.70   270.19


In [5]:
df = df.sort_values(["athlete_id", "session_id"]).reset_index(drop=True)

# 1, 2, 3, ... for each athlete (this is the "session number")
df["session_number"] = df.groupby("athlete_id").cumcount() + 1

print(df.groupby("athlete_id")["session_number"].max().describe().round(1))
df[["athlete_id", "session_id", "session_number"]].head(8)

count    156.0
mean      98.8
std        0.9
min       98.0
25%       98.0
50%       99.0
75%       99.0
max      101.0
Name: session_number, dtype: float64


,athlete_id,session_id,session_number
0,1,1,1
1,1,2,2
2,1,3,3
3,1,4,4
4,1,5,5
5,1,6,6
6,1,7,7
7,1,8,8


In [6]:
g = df.groupby("athlete_id")

# --- Training load: totals, averages and comparison with TODAY ---
for w in [7, 14]:
    # shift(1) first -> only EARLIER sessions are used, never today's
    df[f"load_sum_{w}"] = g["training_load"].transform(
        lambda s: s.shift(1).rolling(w, min_periods=w).sum())
    df[f"load_mean_{w}"] = g["training_load"].transform(
        lambda s: s.shift(1).rolling(w, min_periods=w).mean())
    df[f"load_minus_mean_{w}"] = df["training_load"] - df[f"load_mean_{w}"]
    df[f"load_ratio_{w}"] = df["training_load"] / df[f"load_mean_{w}"]

# --- Averages of the other factors over the previous sessions ---
for col in ["training_intensity", "heart_rate", "sleep_quality",
            "recovery_score", "stress_level"]:
    for w in [7, 14]:
        df[f"{col}_mean_{w}"] = g[col].transform(
            lambda s: s.shift(1).rolling(w, min_periods=3).mean())

df = df.replace([np.inf, -np.inf], np.nan)
print("Total columns now:", df.shape[1])

Total columns now: 29


In [7]:
a = df[df["athlete_id"] == df["athlete_id"].iloc[0]].reset_index(drop=True)
row15 = a[a["session_number"] == 15].iloc[0]

manual_sum7  = a[(a.session_number >= 8) & (a.session_number <= 14)]["training_load"].sum()
manual_sum14 = a[(a.session_number >= 1) & (a.session_number <= 14)]["training_load"].sum()

print("Session 15 -> 7-session sum:  code =", round(row15["load_sum_7"], 2), "| by hand =", round(manual_sum7, 2))
print("Session 15 -> 14-session sum: code =", round(row15["load_sum_14"], 2), "| by hand =", round(manual_sum14, 2))

assert np.isclose(row15["load_sum_7"], manual_sum7)
assert np.isclose(row15["load_sum_14"], manual_sum14)
print("OK: windows use only earlier sessions.")

Session 15 -> 7-session sum:  code = 5664.52 | by hand = 5664.52
Session 15 -> 14-session sum: code = 9996.86 | by hand = 9996.86
OK: windows use only earlier sessions.


In [8]:
print("Correlation between 7-session and 14-session load totals:",
      round(df["load_sum_7"].corr(df["load_sum_14"]), 3))
print("Correlation between 7-session and 14-session load averages:",
      round(df["load_mean_7"].corr(df["load_mean_14"]), 3))

Correlation between 7-session and 14-session load totals: 0.729
Correlation between 7-session and 14-session load averages: 0.729


In [9]:
# Session 15 is the first prediction (the first 14 sessions are the initial history)
model_df = df[df["session_number"] >= 15].copy()

# Chronological split: earlier sessions train, later sessions test
train_df = model_df[model_df["session_number"] <= 75].copy()
test_df  = model_df[model_df["session_number"] >= 76].copy()

print("Prediction rows:", len(model_df))
print("Train rows:", len(train_df), "| Test rows:", len(test_df))

Prediction rows: 13236
Train rows: 9516 | Test rows: 3720


In [10]:
q1, q2 = train_df["fatigue_index"].quantile([1/3, 2/3])
print("Class cut points from training data:", round(q1, 2), round(q2, 2))

def to_class(x):
    if x <= q1:   return 0   # Low
    elif x <= q2: return 1   # Moderate
    else:         return 2   # High

class_names = ["Low", "Moderate", "High"]

train_df["fatigue_class"] = train_df["fatigue_index"].apply(to_class)
test_df["fatigue_class"]  = test_df["fatigue_index"].apply(to_class)

print("\nTrain class counts:\n", train_df["fatigue_class"].value_counts().sort_index())
print("\nTest class counts:\n", test_df["fatigue_class"].value_counts().sort_index())

Class cut points from training data: 47.4 68.77

Train class counts:
 fatigue_class
0    3172
1    3172
2    3172
Name: count, dtype: int64

Test class counts:
 fatigue_class
0    1304
1    1232
2    1184
Name: count, dtype: int64


In [11]:
current_features = ["training_duration", "training_intensity", "training_load",
                    "heart_rate", "sleep_quality", "recovery_score", "stress_level"]

history_features = [c for c in df.columns
                    if c.startswith(("load_sum_", "load_mean_", "load_minus_mean_", "load_ratio_"))
                    or any(c == f"{col}_mean_{w}" for col in
                           ["training_intensity", "heart_rate", "sleep_quality",
                            "recovery_score", "stress_level"] for w in [7, 14])]

features = current_features + history_features

X_train, y_train = train_df[features], train_df["fatigue_class"]
X_test,  y_test  = test_df[features],  test_df["fatigue_class"]

print("Number of input features:", len(features))
print("fatigue_index in features?", "fatigue_index" in features)   # must be False

Number of input features: 25
fatigue_index in features? False


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median")),   # learns medians from training data only
    ("scale", StandardScaler()),                          # puts features on the same scale
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

t0 = time.perf_counter()
logreg.fit(X_train, y_train)
train_wall_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred  = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)
predict_wall_time = time.perf_counter() - t0

print(f"Training wall time:   {train_wall_time:.4f} seconds")
print(f"Prediction wall time: {predict_wall_time:.4f} seconds")

Training wall time:   0.1929 seconds
Prediction wall time: 0.0086 seconds


In [13]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "roc_auc_ovr_macro": roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"),
}

for k, v in metrics.items():
    print(f"{k:20s}: {v:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(cm, index=[f"true_{n}" for n in class_names],
                   columns=[f"pred_{n}" for n in class_names]))

print("\n", classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

accuracy            : 0.6788
balanced_accuracy   : 0.6782
precision_macro     : 0.6789
recall_macro        : 0.6782
f1_macro            : 0.6785
f1_weighted         : 0.6788
roc_auc_ovr_macro   : 0.8522

Confusion matrix (rows = true, columns = predicted):
               pred_Low  pred_Moderate  pred_High
true_Low            978            294         32
true_Moderate       315            669        248
true_High            35            271        878

               precision    recall  f1-score   support

         Low       0.74      0.75      0.74      1304
    Moderate       0.54      0.54      0.54      1232
        High       0.76      0.74      0.75      1184

    accuracy                           0.68      3720
   macro avg       0.68      0.68      0.68      3720
weighted avg       0.68      0.68      0.68      3720



In [14]:
results = pd.DataFrame([{
    "model": "Logistic Regression",
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "n_features": len(features),
    "train_wall_time_sec": round(train_wall_time, 4),
    "predict_wall_time_sec": round(predict_wall_time, 4),
    "total_wall_time_sec": round(train_wall_time + predict_wall_time, 4),
    **{k: round(v, 4) for k, v in metrics.items()},
}])

per_class = pd.DataFrame(classification_report(
    y_test, y_pred, target_names=class_names, output_dict=True, zero_division=0)).T

results.to_csv("/kaggle/working/logistic_regression_metrics.csv", index=False)
per_class.to_csv("/kaggle/working/logistic_regression_per_class.csv")

print(results.T)
print("\nSaved to /kaggle/working/")
print(os.listdir("/kaggle/working"))

                                         0
model                  Logistic Regression
train_rows                            9516
test_rows                             3720
n_features                              25
train_wall_time_sec                 0.1929
predict_wall_time_sec               0.0086
total_wall_time_sec                 0.2015
accuracy                            0.6788
balanced_accuracy                   0.6782
precision_macro                     0.6789
recall_macro                        0.6782
f1_macro                            0.6785
f1_weighted                         0.6788
roc_auc_ovr_macro                   0.8522

Saved to /kaggle/working/
['__notebook__.ipynb', 'logistic_regression_per_class.csv', 'logistic_regression_metrics.csv']
